In [ ]:
import pandas as pd

# Import pipeline
from transcription_pipeline import nuclear_pipeline
from transcription_pipeline import preprocessing_pipeline

from transcription_pipeline import spot_pipeline
from transcription_pipeline import fullEmbryo_pipeline

from transcription_pipeline.spot_analysis import compile_data
from transcription_pipeline.utils import plottable

import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import gc


In [21]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '003'

embryo_list = {
    '001': [
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
    ],
    '002': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],
    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ]
}


In [ ]:
def import_dataset(test_dataset_name, use_filtered=False):
    import_previous_ms2 = os.path.isdir(test_dataset_name + '/collated_dataset')
    if import_previous_ms2:
        print('Reading previous imported dataset')
    else:
        print('No previous dataset import found; importing from scratch')

    dataset = preprocessing_pipeline.DataImport(
        name_folder=test_dataset_name,
        trim_series=True,
        working_storage_mode='zarr',
        import_previous=import_previous_ms2,
        use_filtered=use_filtered,
    )
    if not import_previous_ms2:
        dataset.save()
    return dataset


def import_fullEmbryo_dataset(test_dataset_name):
    import_previous_fullEmbryo = os.path.isdir(test_dataset_name + '/preprocessed_full_embryo')

    if import_previous_fullEmbryo:
        print('Reading previous imported FullEmbryo dataset')
    else:
        print('No previous FullEmbryo dataset import found; importing from scratch')

    FullEmbryo_dataset = preprocessing_pipeline.FullEmbryoImport(
        name_folder=test_dataset_name,
        import_previous=import_previous_fullEmbryo,
    )
    if not import_previous_fullEmbryo:
        FullEmbryo_dataset.save()
    return FullEmbryo_dataset


def get_nuclear_params(dataset, nuclear_channel):
    return dict(
        data=dataset.channels_full_dataset[nuclear_channel],
        global_metadata=dataset.export_global_metadata[nuclear_channel],
        frame_metadata=dataset.export_frame_metadata[nuclear_channel],
        series_splits=dataset.series_splits,
        series_shifts=dataset.series_shifts,
        search_range_um=1.5,
        stitch=False,
        stitch_max_distance=4,
        stitch_max_frame_distance=2,
        client=client,
        keep_futures=False,
    )


def get_spot_params(dataset, spot_channel, Labels, search_range_um, threshold_factor, retrack_after_filter):
    return dict(
        data=dataset.channels_full_dataset[spot_channel],
        global_metadata=dataset.export_global_metadata[spot_channel],
        frame_metadata=dataset.export_frame_metadata[spot_channel],
        labels=Labels,
        expand_distance=3,
        search_range_um=search_range_um,
        retrack_search_range_um=4.5,
        spot_sigma_z_bounds=(0.16, 100),
        spot_sigma_x_y_bounds=(0.052, 100),
        spot_sigmas=[0.86, 0.26, 0.26],
        extract_sigma_multiple=[1,10,10],
        threshold_factor=threshold_factor,
        memory=100,
        retrack_after_filter=retrack_after_filter,
        stitch=True,
        min_track_length=0,
        series_splits=dataset.series_splits,
        series_shifts=dataset.series_shifts,
        keep_bandpass=False,
        keep_futures=False,
        keep_spot_labels=False,
        evaluate=True,
        retrack_by_intensity=False,
        client=client,
        gaussian_fit=False,
    )


def track_import_nuclei(test_dataset_name, dataset, nuclear_channel, spot_channel, redo_tracking=False,
                        folder_name='nuclear_analysis_results'):
    folder_path = test_dataset_name + '/' + folder_name
    import_previous_nuclear = os.path.isdir(folder_path) and os.listdir(folder_path)

    if import_previous_nuclear and not redo_tracking:
        print(f'Reading previous nuclear tracking results (retrack={redo_tracking})')
        nuclear_tracking = nuclear_pipeline.Nuclear()
        nuclear_tracking.read_results(name_folder=test_dataset_name, source_folder=folder_name)

    else:
        if import_previous_nuclear and redo_tracking:
            print(f'Previous nuclear tracking detected. Retracking nuclei (retrack={redo_tracking})')
        else:
            print(f'No previous nuclear tracking results found; importing from scratch (retrack={redo_tracking})')

        nuclear_tracking = nuclear_pipeline.Nuclear(**get_nuclear_params(dataset, nuclear_channel))
        nuclear_tracking.track_nuclei(
            working_memory_mode="zarr",
            working_memory_folder=test_dataset_name + '/' + folder_name,
            trackpy_log_path=test_dataset_name + "trackpy_log",
        )
        nuclear_tracking.save_results(name_folder=test_dataset_name, save_array_as=None, output_folder=folder_name)

    return nuclear_tracking


def track_import_spots(test_dataset_name, dataset, nuclear_channel, spot_channel,
                       redo_tracking=False, use_nuclear_tracking=True,
                       search_range_um=2, threshold_factor=1.5, retrack_after_filter=False,
                       folder_name='spot_analysis_results'):
    folder_path = test_dataset_name + '/' + folder_name
    import_previous_spot = os.path.isdir(folder_path) and os.listdir(folder_path)

    if use_nuclear_tracking:
        nuclear_tracking = track_import_nuclei(
            test_dataset_name, dataset, nuclear_channel=nuclear_channel, spot_channel=spot_channel, redo_tracking=False
        )
        Labels = nuclear_tracking.reordered_labels
    else:
        Labels = None

    if import_previous_spot and not redo_tracking:
        print(f'Load from spot tracking results (retrack={redo_tracking})')
        spot_tracking = spot_pipeline.Spot()
        spot_tracking.read_results(name_folder=test_dataset_name, source_folder=folder_name)

    else:
        if import_previous_spot and redo_tracking:
            print(f'Previous spot tracking detected. Retracking spots (retrack={redo_tracking})')
        else:
            print(f'No previous spot tracking results found; importing from scratch (retrack={redo_tracking})')

        spot_tracking = spot_pipeline.Spot(
            **get_spot_params(dataset, spot_channel, Labels, search_range_um, threshold_factor, retrack_after_filter)
        )
        spot_tracking.extract_spot_traces(
            working_memory_folder=test_dataset_name + '/' + folder_name,
            stitch=True,
            retrack_after_filter=retrack_after_filter,
            trackpy_log_path=test_dataset_name + '/trackpy_log',
            verbose=True,
            filter_by_sigma=False,
        )
        spot_tracking.save_results(name_folder=test_dataset_name, save_array_as=None, output_folder=folder_name)

    return spot_tracking


In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(
    host="localhost",
    #scheduler_port=37763,
    threads_per_worker=1,
    n_workers=14,
    memory_limit="6GB",
)

client = Client(cluster)

print(client)

In [ ]:
print(client.dashboard_link)


In [ ]:
client.restart()


In [ ]:
nuclear_channel = 1
spot_channel = 0


In [ ]:
gc.collect()

for embryo_index in range(len(embryo_list[key])):
    test_dataset_name = dataset_folder + embryo_list[key][embryo_index]
    print(f'\n{"=" * 60}')
    print(f'Dataset Path: {test_dataset_name}')
    print(f'{"=" * 60}\n')

    try:
        gc.disable()
        # Load the dataset with time-filtered data
        dataset = import_dataset(test_dataset_name, use_filtered=True)

        # Load the full embryo dataset
        FullEmbryo_dataset = import_fullEmbryo_dataset(test_dataset_name)
        gc.enable()
        gc.collect()

        # Track and import nuclei with time-filtered data
        nuclear_tracking = track_import_nuclei(
            test_dataset_name, dataset,
            nuclear_channel=nuclear_channel,
            spot_channel=spot_channel,
            redo_tracking=False,
            folder_name='nuclear_analysis_results_time_filtered',
        )

        # Clear memory
        del nuclear_tracking
        gc.collect()

        # Track and import spots with time-filtered data
        spot_tracking = track_import_spots(
            test_dataset_name, dataset,
            nuclear_channel=nuclear_channel,
            spot_channel=spot_channel,
            redo_tracking=True,
            use_nuclear_tracking=False,
            search_range_um=5,
            threshold_factor=1.8,
            retrack_after_filter=False,
            folder_name='spot_analysis_results_time_filtered',
        )

        # Clear memory before next iteration
        del spot_tracking
        del dataset
        del FullEmbryo_dataset
        gc.collect()

        print(f'\nSuccessfully processed: {test_dataset_name}\n')

    except Exception as e:
        print(f'\nERROR processing {test_dataset_name}:')
        print(f'{type(e).__name__}: {str(e)}\n')
        # Try to recover by restarting client
        gc.enable()
        gc.collect()
        try:
            client.restart()
        except:
            pass


In [ ]:
test_dataset_name = dataset_folder + embryo_list[key][0]
print(f'\n{"=" * 60}')
print(f'Dataset Path: {test_dataset_name}')
print(f'{"=" * 60}\n')

gc.collect()

gc.disable()
dataset = import_dataset(test_dataset_name, use_filtered=True)
FullEmbryo_dataset = import_fullEmbryo_dataset(test_dataset_name)
gc.enable()
gc.collect()

nuclear_tracking = track_import_nuclei(
    test_dataset_name, dataset,
    nuclear_channel=nuclear_channel,
    spot_channel=spot_channel,
    redo_tracking=False,
    folder_name='nuclear_analysis_results_time_filtered',
)

del nuclear_tracking
gc.collect()

spot_tracking = track_import_spots(
    test_dataset_name, dataset,
    nuclear_channel=nuclear_channel,
    spot_channel=spot_channel,
    redo_tracking=False,
    use_nuclear_tracking=False,
    search_range_um=5,
    threshold_factor=2,
    retrack_after_filter=False,
    folder_name='spot_analysis_results_time_filtered',
)

del spot_tracking
del dataset
del FullEmbryo_dataset
gc.collect()

print(f'\nSuccessfully processed: {test_dataset_name}\n')


In [ ]:
# Load nuclear tracking (time_filtered)
nuclear_tracking = nuclear_pipeline.Nuclear()
nuclear_tracking.read_results(name_folder=test_dataset_name, source_folder='nuclear_analysis_results_time_filtered')
nuclear_df = nuclear_tracking.mitosis_dataframe
nuclear_df.head()

In [22]:
import numpy as np

mpl.use('TkAgg')

# Loop over all embryos in the given key
for embryo_index in range(len(embryo_list[key])):
    test_dataset_name = dataset_folder + embryo_list[key][embryo_index]

    print(f'\n{"=" * 80}')
    print(f'Processing Embryo {embryo_index + 1}/{len(embryo_list[key])} for key "{key}"')
    print(f'Dataset: {embryo_list[key][embryo_index]}')
    print(f'{"=" * 80}\n')

    # Load the dataset
    print('→ Loading dataset...')
    dataset = import_dataset(test_dataset_name)

    # Load the full embryo dataset
    print('→ Loading full embryo dataset...')
    FullEmbryo_dataset = import_fullEmbryo_dataset(test_dataset_name)

    # Load spot tracking (time_filtered)
    print('→ Loading spot tracking results...')
    spot_tracking = spot_pipeline.Spot()
    spot_tracking.read_results(name_folder=test_dataset_name, source_folder='spot_analysis_results_time_filtered')
    spot_df = spot_tracking.spot_dataframe

    # Load nuclear tracking (time_filtered)
    print('→ Loading nuclear tracking results...')
    nuclear_tracking = nuclear_pipeline.Nuclear()
    nuclear_tracking.read_results(name_folder=test_dataset_name, source_folder='nuclear_analysis_results_time_filtered')
    nuclear_df = nuclear_tracking.mitosis_dataframe

    # Remove spots that were not detected
    detected_spots = spot_df[spot_df["particle"] != 0]

    # Remove nuclei that were not detected
    detected_nuclei = nuclear_df[nuclear_df["particle"] != 0]

    # Compile spot traces
    print('→ Compiling spot traces...')
    compiled_dataframe_spots = compile_data.compile_traces(
        spot_tracking_dataframe=detected_spots,
        compile_columns_spot=[
            "frame",
            "t_s",
            "intensity_from_neighborhood",
            "intensity_std_error_from_neighborhood",
            "x",
            "y"
        ],
        nuclear_tracking_dataframe=None,
    )

    # Compile nuclear traces
    print('→ Compiling nuclear traces...')
    compiled_dataframe_nuclear = compile_data.compile_nuclei(
        nuclear_tracking_dataframe=detected_nuclei,
        compile_columns_nuclear=[
            "frame",
            "t_s",
            "x_pixel",
            "y_pixel",
        ]
    )

    # Rename 'x_pixel' and 'y_pixel' to 'x' and 'y'
    compiled_dataframe_nuclear.rename(columns={'x_pixel': 'x', 'y_pixel': 'y'}, inplace=True)

    # Filter spots and nuclei by trace length (>= 100 frames)
    print('→ Filtering traces by length (>= 100 frames)...')
    compiled_dataframe_spots_filtered = compiled_dataframe_spots[
        compiled_dataframe_spots['frame'].apply(lambda x: len(x) >= 100)
    ]

    compiled_dataframe_nuclear_filtered = compiled_dataframe_nuclear[
        compiled_dataframe_nuclear['frame'].apply(lambda x: len(x) >= 100)
    ]

    # Load the full embryo dataset with previously calculated AP points
    print('→ Loading AP axis information...')
    fullEmbryo = fullEmbryo_pipeline.FullEmbryo(test_dataset_name, FullEmbryo_dataset, dataset,
                                                his_channel=nuclear_channel)
    fullEmbryo.find_ap_axis(make_plots=False, ap_method='maxf', sigma=10, radius=5,
                            load_previous=True, save_results=True)

    # Run xy_to_ap on filtered spot positions
    print('→ Converting spot positions to AP coordinates...')
    compiled_dataframe_spots_filtered = fullEmbryo.xy_to_ap(compiled_dataframe_spots_filtered)

    # Run xy_to_ap on filtered nuclei positions
    print('→ Converting nuclear positions to AP coordinates...')
    compiled_dataframe_nuclear_filtered = fullEmbryo.xy_to_ap(compiled_dataframe_nuclear_filtered)

    # Calculate number of spots and nuclei in each AP bin using filtered dataframes
    print('→ Calculating AP bin statistics...')
    ap_bins = np.linspace(0, 1, 41)  # 40 bins from 0 to 1
    spot_counts = np.histogram([np.median(ap) for ap in compiled_dataframe_spots_filtered['ap']], bins=ap_bins)[0]
    nuclei_counts = np.histogram([np.median(ap) for ap in compiled_dataframe_nuclear_filtered['ap']], bins=ap_bins)[0]

    # Create summary dataframe
    ap_bin_centers = (ap_bins[:-1] + ap_bins[1:]) / 2
    summary_df = pd.DataFrame({
        'ap_bin_center': ap_bin_centers,
        'num_spots': spot_counts,
        'num_nuclei': nuclei_counts
    })

    # Get all unique frames
    all_frames = sorted(set([frame for frames in compiled_dataframe_nuclear_filtered['frame'] for frame in frames]))

    # Initialize list to store time-resolved dataframes
    print('→ Calculating time-resolved fraction active...')
    fraction_active_time_records = []

    for frame_idx in all_frames:
        # Get AP positions for this frame for each track
        spot_ap_frame = []
        for i, frames in enumerate(compiled_dataframe_spots_filtered['frame']):
            if frame_idx in frames:
                frame_position = list(frames).index(frame_idx)
                spot_ap_frame.append(compiled_dataframe_spots_filtered['ap'].iloc[i][frame_position])

        nuclear_ap_frame = []
        for i, frames in enumerate(compiled_dataframe_nuclear_filtered['frame']):
            if frame_idx in frames:
                frame_position = list(frames).index(frame_idx)
                nuclear_ap_frame.append(compiled_dataframe_nuclear_filtered['ap'].iloc[i][frame_position])

        # Calculate counts for this frame
        spot_counts_frame = np.histogram(spot_ap_frame, bins=ap_bins)[0]
        nuclei_counts_frame = np.histogram(nuclear_ap_frame, bins=ap_bins)[0]

        # Calculate fraction active (avoid division by zero)
        fraction_frame = np.divide(spot_counts_frame, nuclei_counts_frame,
                                   where=nuclei_counts_frame > 0,
                                   out=np.zeros_like(spot_counts_frame, dtype=float))

        # Store results for this frame
        for bin_idx, (ap_bin, fraction) in enumerate(zip(ap_bin_centers, fraction_frame)):
            fraction_active_time_records.append({
                'frame': frame_idx,
                'ap_bin': ap_bin,
                'fraction_active': fraction
            })

    # Create dataframe from records
    fraction_active_time_df = pd.DataFrame(fraction_active_time_records)

    # Save results
    print('→ Saving compiled dataframes...')
    compiled_dataframe_spots.to_pickle(test_dataset_name + '/compiled_dataframe_spots_time_filtered.pkl')
    compiled_dataframe_nuclear.to_pickle(test_dataset_name + '/compiled_dataframe_nuclei_time_filtered.pkl')
    summary_df.to_pickle(test_dataset_name + '/fraction_active_time_filtered.pkl')
    fraction_active_time_df.to_pickle(test_dataset_name + '/fraction_active_over_time.pkl')

    print(f'\n✓ Completed processing embryo {embryo_index + 1}')



Processing Embryo 1/5 for key "003"
Dataset: test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01

→ Loading dataset...
Reading previous imported dataset
→ Loading full embryo dataset...
Reading previous imported FullEmbryo dataset
→ Loading spot tracking results...
→ Loading nuclear tracking results...
→ Compiling spot traces...
→ Compiling nuclear traces...
→ Filtering traces by length (>= 100 frames)...
→ Loading AP axis information...
Previous AP points loaded.
AP and AP90 axes defined.
Anterior Point: (930, 266)
Posterior Point: (76, 211)
AP Angle: 3.69 degrees
AP distance: 855.77 pixels
AP90 distance: 358.74 pixels
→ Converting spot positions to AP coordinates...
→ Converting nuclear positions to AP coordinates...
→ Calculating AP bin statistics...
→ Calculating time-resolved fraction active...
→ Saving compiled dataframes...

✓ Completed processing embryo 1

Processing Embryo 2/5 for key "003"
Dataset: test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(tra